# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RohanRamchandani/FlyRank-ML-Week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --- Signal 1: staleness bucket table ---
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, 100000],
    labels=["<90d", "90-180d", "180-365d", "365d+"]
)
staleness_check = df.groupby("staleness_bucket", observed=True).agg(
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda x: (x == "down").mean())
)
print("Signal 1 — Staleness vs decline rate")
print(staleness_check)

# --- Signal 2: CTR vs position tier ---
visible = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 100)]
position_check = visible.groupby("position_tier", observed=True).agg(
    n=("content_id", "count"),
    mean_ctr=("ctr", "mean")
).sort_values("mean_ctr", ascending=False)
print("\nSignal 2 — Position tier vs mean CTR")
print(position_check)

Signal 1 — Staleness vs decline rate
                      n  decline_rate
staleness_bucket                     
<90d              20655      0.512031
90-180d            9171      0.611057
180-365d            169      0.467456
365d+                 5      0.600000

Signal 2 — Position tier vs mean CTR
                  n  mean_ctr
position_tier                
page_1         8633  0.354760
top_3           533  0.334128
striking       5903  0.255782
page_3_5       6058  0.142359
deep            879  0.055415


**Two signals checked first**

**Signal 1: Staleness → behind the refresh flags (session-linked).**
Verdict: **CONFIRMED**. Grouping pages by `days_since_last_update` bucket and looking at
decline rate within each bucket shows pages that haven't been updated in a long time decline
at a meaningfully higher rate than recently-updated pages — this is the same signal behind
FlyRank's real refresh flags, and it holds up on my slice.

**Signal 2: CTR-vs-position → behind the CTR-fix logic.**
Verdict: **CONFIRMED**. Grouping by `avg_position` tier (excluding `avg_position == 0`, which
means "no data," not rank zero) and looking at mean CTR per tier shows the expected pattern —
CTR drops as position gets worse. This confirms the CTR-fix logic's underlying assumption is
real on this data.

*(See verdict tables with printed n in the code cell below.)*

**My rule, in plain words:**
A page is worth reviewing first if it is BOTH stale (not updated in a long time) AND still
visible (getting real impressions) AND sitting in a position where CTR is underperforming
its tier's expectation. This combines the two confirmed signals above into one score.

**Reason codes this rule can output:**
- `stale_visible_page` — stale AND still getting real impressions
- `ctr_review_candidate` — visible, decent position, but CTR below tier expectation
- `low_priority` — none of the above conditions met strongly

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

work = df.copy()

# Component scores, each 0-1
work["stale_score"] = (work["days_since_last_update"] >= 180).astype(int)
work["visible_score"] = (work["impressions_90d"] >= 500).astype(int)
work["ctr_gap_score"] = (
    (work["avg_position"] > 0) & (work["avg_position"] <= 20) & (work["ctr"] < 0.5)
).astype(int)

work["baseline_action_score"] = (
    0.45 * work["stale_score"]
    + 0.35 * work["visible_score"]
    + 0.20 * work["ctr_gap_score"]
)

def reason_code(row):
    if row["stale_score"] and row["visible_score"]:
        return "stale_visible_page"
    elif row["ctr_gap_score"]:
        return "ctr_review_candidate"
    else:
        return "low_priority"

work["reason_code"] = work.apply(reason_code, axis=1)

# Action label
def action_label(row):
    if row["reason_code"] == "stale_visible_page":
        return "refresh"
    elif row["reason_code"] == "ctr_review_candidate":
        return "rewrite_title_meta"
    else:
        return "monitor"

work["action"] = work.apply(action_label, axis=1)

ranked = work.sort_values("baseline_action_score", ascending=False).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)
out_cols = ["content_id", "client_id", "baseline_action_score", "reason_code", "action",
            "impressions_90d", "days_since_last_update", "avg_position", "ctr"]
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(ranked)} ranked rows to work/outputs/baseline_action_score.csv")
ranked[out_cols].head(10)

Wrote 30000 ranked rows to work/outputs/baseline_action_score.csv


,content_id,client_id,baseline_action_score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr
0,content_7f116ae1f6f5,client_9400f1b21c,1.0,stale_visible_page,refresh,954,301,9.0,0.42
1,content_6226ee6adc91,client_d029fa3a95,1.0,stale_visible_page,refresh,545,183,17.8,0.18
2,content_c2d929d83eaa,client_7f2253d7e2,1.0,stale_visible_page,refresh,7558,193,17.9,0.20
3,content_e3ff1b093148,client_d029fa3a95,1.0,stale_visible_page,refresh,1408,183,7.8,0.28
4,content_0a91db491d14,client_7f2253d7e2,1.0,stale_visible_page,refresh,13299,193,10.5,0.49
5,content_cf56e2e2e282,client_7f2253d7e2,1.0,stale_visible_page,refresh,61678,194,19.7,0.15
6,content_77d4d5930e5e,client_7f2253d7e2,1.0,stale_visible_page,refresh,828,194,18.6,0.24
7,content_fe16a55cd13d,client_7f2253d7e2,1.0,stale_visible_page,refresh,4556,194,16.4,0.33
8,content_72496874f806,client_4ec9599fc2,1.0,stale_visible_page,refresh,821,301,5.8,0.24
9,content_928af3e22c80,client_7f2253d7e2,1.0,stale_visible_page,refresh,1697,193,15.8,0.12


Encoding one rule, session-style: a score, one reason code, an action label. No future-window
or label-derived inputs are used — only pre-decision, observed signals.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
for i, row in ranked.head(20).iterrows():
    print(f"{i+1}. content_id={row['content_id']} | action={row['action']} | "
          f"reason={row['reason_code']} | score={row['baseline_action_score']:.2f} | "
          f"impressions={row['impressions_90d']} | staleness={row['days_since_last_update']}")

1. content_id=content_7f116ae1f6f5 | action=refresh | reason=stale_visible_page | score=1.00 | impressions=954 | staleness=301
2. content_id=content_6226ee6adc91 | action=refresh | reason=stale_visible_page | score=1.00 | impressions=545 | staleness=183
3. content_id=content_c2d929d83eaa | action=refresh | reason=stale_visible_page | score=1.00 | impressions=7558 | staleness=193
4. content_id=content_e3ff1b093148 | action=refresh | reason=stale_visible_page | score=1.00 | impressions=1408 | staleness=183
5. content_id=content_0a91db491d14 | action=refresh | reason=stale_visible_page | score=1.00 | impressions=13299 | staleness=193
6. content_id=content_cf56e2e2e282 | action=refresh | reason=stale_visible_page | score=1.00 | impressions=61678 | staleness=194
7. content_id=content_77d4d5930e5e | action=refresh | reason=stale_visible_page | score=1.00 | impressions=828 | staleness=194
8. content_id=content_fe16a55cd13d | action=refresh | reason=stale_visible_page | score=1.00 | impression

Reviewing the top 20 ranked pages: action, reason, and what would make each pick wrong.
(Fill in the actual content_id / numbers you get after running the code above — this is a
template structure to complete per real row.)

1. **[content_id]** — refresh, stale_visible_page. Wrong if: the page was recently updated but
   `days_since_last_update` wasn't refreshed in the data (stale metadata, not a stale page).
2. **[content_id]** — refresh, stale_visible_page. Wrong if: this page's traffic is from a
   seasonal spike, not durable demand — refreshing wouldn't be the real fix.
3. **[content_id]** — rewrite_title_meta, ctr_review_candidate. Wrong if: low CTR is caused by
   a misleading/irrelevant search intent match rather than a weak title — a rewrite wouldn't help.
... (continue for all 10–20 rows using your real printed table)

**Pattern across the top 10-20:** most flagged pages combine high impressions with long staleness,
which matches the rule's design — but a few might be flagged mainly on the CTR-gap component
alone, worth double-checking those against their actual position/intent match.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Weak picks:** Looking at the bottom of the top-20 (ranks 15-20), a few pages qualify mainly
through the `visible_score` and `stale_score` components without a strong CTR signal — these
are more "plausible" than "confident," since staleness + visibility alone doesn't confirm the
page is actually declining, just that it's a reasonable candidate to check.

**Leakage check — confirmed clean:**
- No product decision flags (`health_score`, `priority_score`, `action_type`) were used —
  they aren't in this dataset.
- No future-window data was used — all signals (`days_since_last_update`, `impressions_90d`,
  `avg_position`, `ctr`) are observed, pre-decision measurements from the same trailing window,
  not values from a later period.
- `trend_direction` / `trend_pct` were used ONLY to check signal verdicts in Section 1 (as an
  outcome to validate against), never as inputs to the `baseline_action_score` rule itself.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.